# Radiomics Feature Extraction - Head and Neck Cancer

**Objective:** Extract quantitative features from CT/MRI images for cancer classification

**Workflow:**
1. Load medical images (DICOM/NIfTI)
2. Load segmentation masks (ROI)
3. Extract radiomic features using PyRadiomics
4. Save features for machine learning

**Dataset:** TCIA Head-Neck Cancer (or sample data provided)

## Setup & Installation

Run this cell first to install required libraries

In [37]:
# Install required libraries
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git
!pip install -q SimpleITK pandas numpy matplotlib seaborn scikit-learn

# For Google Colab - mount Google Drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully")
except:
    print("Not running in Google Colab - continuing with local environment")

  Cloning https://github.com/AIM-Harvard/pyradiomics.git to /tmp/pip-req-build-8cv2y4rs
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-8cv2y4rs
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully


In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import seaborn as sns
import radiomics
from radiomics import featureextractor
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"PyRadiomics version: {radiomics.__version__}")

## 1. Understanding Radiomics Features

Radiomics extracts quantitative features from medical images:

1. **First-Order Statistics** (19 features)
   - Mean, Standard Deviation, Skewness, Kurtosis
   - Energy, Entropy, Minimum, Maximum

2. **Shape Features** (14-16 features)
   - Volume, Surface Area, Sphericity, Compactness
   - Elongation, Flatness

3. **Texture Features** (GLCM, GLRLM, GLSZM, GLDM, NGTDM)
   - ~70 features describing spatial patterns

4. **Wavelet Features**
   - Features extracted at different frequencies

## 2. Configuration Setup

Configure PyRadiomics parameters

In [ ]:
# Configuration for feature extraction
settings = {
    'binWidth': 25,                    # Bin width for discretization
    'interpolator': 'sitkBSpline',       # Interpolation method
    'resampledPixelSpacing': None,     # No resampling
    'label': 1,                        # Label value for ROI
    'normalize': True,                 # Normalize image
    'normalizeScale': 100,             # Scale after normalization
    'resegmentMode': 'absolute',       # Resegmentation mode
    'resegmentRange': [-1000, 3000],   # CT intensity range
    'removeOutliers': None,            # Outlier removal
}

# Feature classes to extract
feature_classes = {
    'shape': True,
    'firstorder': True,
    'glcm': True,                      # Gray Level Co-occurrence Matrix
    'glrlm': True,                     # Gray Level Run Length Matrix
    'glszm': True,                     # Gray Level Size Zone Matrix
    'gldm': True,                      # Gray Level Dependence Matrix
    'ngtdm': True,                     # Neighboring Gray Tone Difference Matrix
}

# Initialize feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
for feature_class in feature_classes:
    extractor.enableFeatureClassByName(feature_class)

print("✓ Feature extractor configured")
print(f"Enabled feature classes: {list(feature_classes.keys())}")

## 3. Load Sample Data

Choose one of the following options to obtain an image and mask for extraction:

**Option A: PyRadiomics Built-in Test Case** (Recommended for beginners)
- Downloads a real clinical sample (e.g., 'brain1', 'breast1') directly from the library repository.

**Option B: Synthetic Data** (Fast, no internet required)
- Generates a simple 3D cube with a simulated "tumor" locally.

**Option C: TCIA (The Cancer Imaging Archive)**
- For advanced projects using real patient datasets (requires manual download).

In [ ]:
# Choose your sample data option: 'built-in' or 'synthetic'
DATA_OPTION = 'built-in' 

def get_data(option='built-in'):
    if option == 'built-in':
        print("Downloading/Loading PyRadiomics test case 'brain1'...")
        try:
            image_path, mask_path = radiomics.getTestCase('brain1')
            return image_path, mask_path
        except Exception as e:
            print(f"Download failed: {e}. Falling back to synthetic.")
            option = 'synthetic'
            
    if option == 'synthetic':
        import os
        import SimpleITK as sitk
        os.makedirs('data/sample', exist_ok=True)
        image_path = 'data/sample/image.nii.gz'
        mask_path = 'data/sample/mask.nii.gz'
        
        if not os.path.exists(image_path):
            img_data = np.random.normal(0, 100, (64, 64, 64)).astype(np.float32)
            img_data[20:40, 20:40, 20:40] += 500
            sitk.WriteImage(sitk.GetImageFromArray(img_data), image_path)
            
            mask_data = np.zeros((64, 64, 64), dtype=np.uint8)
            mask_data[20:40, 20:40, 20:40] = 1
            sitk.WriteImage(sitk.GetImageFromArray(mask_data), mask_path)
        return image_path, mask_path

image_path, mask_path = get_data(DATA_OPTION)
print(f"✓ Image path: {image_path}")
print(f"✓ Mask path: {mask_path}")

## 4. Extract Features

Extract features from the chosen sample case

In [ ]:
# Execute extraction
print("Extracting features...")
result = extractor.execute(image_path, mask_path)

# Filter for only the radiomics features (ignore metadata)
features = {k: v for k, v in result.items() if not k.startswith('diagnostics')}

print(f"✓ Extracted {len(features)} features")

# Show top 10 features
for i, (key, value) in enumerate(list(features.items())[:10]):
    print(f"{key}: {value:.4f}")

In [ ]:
# Convert extracted features to a clean pandas Series and visualize
feature_series = pd.Series(features).sort_values(ascending=False)

# Convert numpy scalars/arrays to python floats for plotting (or NaN for non-scalar)
def to_scalar(v):
    arr = np.asarray(v)
    if arr.size == 1:
        return float(arr.item())
    return np.nan

feature_series = feature_series.map(to_scalar).dropna()

# Plot top 20 features
top_n = 20
plt.figure(figsize=(14, 6))
sns.barplot(x=feature_series.index[:top_n], y=feature_series.values[:top_n], palette="viridis")
plt.xticks(rotation=90)
plt.ylabel("Feature value")
plt.title(f"Top {top_n} extracted radiomics features")
plt.tight_layout()
plt.show()

# Optional: visualize grouped feature classes
for prefix in ['original_shape', 'original_firstorder', 'original_glcm']:
    group = feature_series[feature_series.index.str.startswith(prefix)]
    if len(group) > 0:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group.index, y=group.values, palette="magma")
        plt.xticks(rotation=90)
        plt.title(f"{prefix} features")
        plt.tight_layout()
        plt.show()

## 5. Visualizing Textures

Visualize the image and mask

In [ ]:
def plot_slice(image_path, mask_path, slice_idx=None):
    img_obj = sitk.ReadImage(image_path)
    mask_obj = sitk.ReadImage(mask_path)
    
    img = sitk.GetArrayFromImage(img_obj)
    mask = sitk.GetArrayFromImage(mask_obj)
    
    # Auto-find middle slice if not specified
    if slice_idx is None:
        slice_idx = img.shape[0] // 2
        # If using mask, find slice with most mask pixels
        mask_sums = np.sum(mask, axis=(1, 2))
        if np.any(mask_sums > 0):
            slice_idx = np.argmax(mask_sums)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.title(f"Image Slice {slice_idx}")
    plt.imshow(img[slice_idx], cmap='gray')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"Mask Overlay")
    plt.imshow(img[slice_idx], cmap='gray')
    plt.imshow(mask[slice_idx], cmap='jet', alpha=0.4)
    plt.axis('off')
    
    plt.show()

plot_slice(image_path, mask_path)

## 6. Batch Extraction

In a real project, you would extract features from many patients

In [ ]:
# This is a placeholder for batch extraction logic
def batch_extract(patient_list, extractor):
    all_features = []
    for patient_id in tqdm(patient_list):
        # In real life: load patient image and mask paths
        # features = extractor.execute(p_img, p_mask)
        # all_features.append(features)
        pass
    return pd.DataFrame(all_features)

print("Batch extraction template ready")

## 7. Next Steps

Now that you can extract features, proceed to `02_machine_learning.ipynb` to build models!